# Municipal Legal Code Exploratory Analysis Demo

This notebook demonstrates the end-to-end agentic research system for conducting exploratory data analysis of municipal legal code ordinances.

In [ ]:
# Install required packages if running in Colab
import sys
import os

# Add the src directory to the Python path
if '../src' not in sys.path:
    sys.path.insert(0, '../src')

# Import the municipal research system
from municipal_research.agents import ResearchAgent
from municipal_research.data_collection import MunicipalDataCollector
from municipal_research.analysis import LegalCodeAnalyzer, LegalCodeVisualizer
from municipal_research.utils import setup_logging, ConfigManager

In [ ]:
# Set up logging
setup_logging(log_level="INFO")

# Load configuration
config_manager = ConfigManager('../config/config.json')
config_manager.create_directories()

## 1. Initialize the Research System

First, let's initialize our agentic research system with the research agent.

In [ ]:
# Initialize research agent
research_config = {
    'data_dir': '../data',
    'visualization_dir': '../visualizations',
    'reports_dir': '../reports'
}

agent = ResearchAgent(research_config)
print("✅ Research Agent initialized successfully")

## 2. Data Collection

Let's collect sample municipal legal codes from California, Georgia, Florida, and Texas.

In [ ]:
# Define data collection task
collection_task = {
    'type': 'collect_sample_data',
    'parameters': {
        'max_jurisdictions_per_state': 2,  # Collect from 2 cities per state
        'max_codes_per_jurisdiction': 3,   # Collect 3 codes per city
        'save_data': True,
        'filename_prefix': 'demo_data'
    }
}

print("🚀 Starting data collection...")
collection_result = agent.data_agent.execute_task(collection_task)

if collection_result['status'] == 'success':
    print(f"✅ Data collection completed!")
    print(f"Total codes collected: {collection_result['total_codes']}")
    
    # Display summary by state
    summary = collection_result['summary']
    print("\nCodes collected by state:")
    for state, info in summary['states'].items():
        print(f"  {state}: {info['code_count']} codes from {info['jurisdiction_count']} jurisdictions")
else:
    print(f"❌ Data collection failed: {collection_result.get('error')}")

## 3. Exploratory Data Analysis

Now let's analyze the collected legal codes using our analysis agent.

In [ ]:
# Define analysis task
analysis_task = {
    'type': 'analyze_legal_codes',
    'data': collection_result['data'],
    'parameters': {
        'generate_visualizations': True,
        'visualization_dir': '../visualizations'
    }
}

print("🔍 Starting analysis...")
analysis_result = agent.analysis_agent.execute_task(analysis_task)

if analysis_result['status'] == 'success':
    print("✅ Analysis completed!")
    
    # Display analysis summary
    analysis = analysis_result['analysis']
    
    print("\nAnalysis Overview:")
    overview = analysis.get('overview', {})
    print(f"  Total codes: {overview.get('total_codes', 0)}")
    print(f"  States: {overview.get('total_states', 0)}")
    print(f"  Jurisdictions: {overview.get('total_jurisdictions', 0)}")
    
    # Text statistics
    if 'text_statistics' in analysis:
        stats = analysis['text_statistics']['word_count_stats']
        print(f"\nText Statistics:")
        print(f"  Average words per code: {stats['mean']:.0f}")
        print(f"  Total words: {stats['total']:,}")
        print(f"  Longest code: {stats['max']:,} words")
        
else:
    print(f"❌ Analysis failed: {analysis_result.get('error')}")

## 4. Topic Analysis

Let's examine what topics are most common in the collected legal codes.

In [ ]:
if 'topic_analysis' in analysis:
    topics = analysis['topic_analysis']
    
    print("📊 Topic Analysis Results:")
    print("\nMost common topics:")
    
    topic_freq = topics.get('topic_frequencies', {})
    sorted_topics = sorted(topic_freq.items(), key=lambda x: x[1], reverse=True)
    
    for i, (topic, frequency) in enumerate(sorted_topics[:8], 1):
        topic_name = topic.replace('_', ' ').title()
        print(f"  {i}. {topic_name}: {frequency} mentions")
        
    # Display topic distribution
    print("\nCodes by topic category:")
    topic_dist = topics.get('topic_distribution', {})
    for topic, count in sorted(topic_dist.items(), key=lambda x: x[1], reverse=True)[:5]:
        topic_name = topic.replace('_', ' ').title()
        print(f"  {topic_name}: {count} codes")

## 5. Readability Analysis

Let's analyze how readable these legal codes are for the general public.

In [ ]:
if 'readability_analysis' in analysis:
    readability = analysis['readability_analysis']
    
    print("📖 Readability Analysis:")
    
    if 'average_score' in readability:
        avg_score = readability['average_score']
        interpretation = readability['interpretation']
        
        print(f"\nAverage readability score: {avg_score:.1f}")
        print(f"Interpretation: {interpretation}")
        
        print(f"\nScore range: {readability['min_score']:.1f} - {readability['max_score']:.1f}")
        print(f"Standard deviation: {readability['std_score']:.1f}")
        
        # Show readability by jurisdiction (top 5)
        by_jurisdiction = readability.get('by_jurisdiction', [])
        if by_jurisdiction:
            print("\nReadability by jurisdiction (sample):")
            sorted_jurisdictions = sorted(by_jurisdiction, key=lambda x: x['score'], reverse=True)
            for jurisdiction in sorted_jurisdictions[:5]:
                print(f"  {jurisdiction['jurisdiction']}, {jurisdiction['state']}: {jurisdiction['score']:.1f}")
else:
    print("⚠️  Readability analysis not available (requires textstat library)")

## 6. State-by-State Comparison

Let's compare the characteristics of legal codes across different states.

In [ ]:
if 'comparative_analysis' in analysis:
    comparison = analysis['comparative_analysis']
    
    print("🗺️  State-by-State Comparison:")
    print()
    
    for state, metrics in comparison.items():
        print(f"{state}:")
        print(f"  Codes collected: {metrics['code_count']}")
        print(f"  Jurisdictions: {metrics['jurisdiction_count']}")
        print(f"  Average words per code: {metrics['avg_word_count']:.0f}")
        print(f"  Average readability: {metrics['avg_readability']:.1f}")
        
        # Show top terms for this state
        if 'common_terms' in metrics:
            terms = list(metrics['common_terms'].keys())[:3]
            if terms:
                print(f"  Common terms: {', '.join(terms)}")
        print()

## 7. Generate Complete Report

Finally, let's generate a comprehensive research report.

In [ ]:
# Generate comprehensive report
report_task = {
    'type': 'generate_report',
    'analysis_results': analysis,
    'collection_summary': collection_result['summary']
}

print("📄 Generating comprehensive report...")
report_result = agent.execute_task(report_task)

if report_result['status'] == 'success':
    report = report_result['report']
    
    print("✅ Report generated successfully!")
    print()
    print("=" * 60)
    print(report['title'])
    print("=" * 60)
    print(f"Generated: {report['generated_at']}")
    print()
    
    print("EXECUTIVE SUMMARY:")
    print(report['executive_summary'])
    print()
    
    print("KEY FINDINGS:")
    findings = report.get('findings', [])
    for i, finding in enumerate(findings, 1):
        print(f"{i}. {finding}")
    print()
    
    print("RECOMMENDATIONS:")
    recommendations = report.get('recommendations', [])
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")
        
else:
    print(f"❌ Report generation failed: {report_result.get('error')}")

## 8. Visualizations

If visualizations were generated, let's display information about them.

In [ ]:
# Check for generated visualizations
visualizations = analysis_result.get('visualizations', {})

if visualizations:
    print("📊 Generated Visualizations:")
    print()
    
    for viz_type, path in visualizations.items():
        viz_name = viz_type.replace('_', ' ').title()
        print(f"  {viz_name}: {path}")
        
    print(f"\n📁 All visualizations saved to: ../visualizations/")
    
    # Display one of the visualizations if running in a notebook environment
    try:
        from IPython.display import Image, display
        import os
        
        # Show the codes by state chart if it exists
        codes_by_state_path = visualizations.get('codes_by_state')
        if codes_by_state_path and os.path.exists(codes_by_state_path):
            print("\nCodes by State Visualization:")
            display(Image(codes_by_state_path))
            
    except ImportError:
        print("\n📝 Note: Install IPython to display visualizations in this notebook")
        
else:
    print("⚠️  No visualizations were generated")

## Summary

This demo showcased the end-to-end agentic research system for municipal legal code analysis:

1. **Automated Data Collection**: Collected legal codes from multiple states and jurisdictions
2. **Text Analysis**: Analyzed word counts, readability, and common terms
3. **Topic Analysis**: Identified common themes in municipal legal codes
4. **Comparative Analysis**: Compared characteristics across different states
5. **Visualization**: Generated charts and graphs to illustrate findings
6. **Report Generation**: Created a comprehensive research report

The system demonstrates how agentic AI can automate complex research workflows, making large-scale legal text analysis more accessible and efficient.